<a href="https://colab.research.google.com/github/gabiuxo/Algoritmos-de-Aprendizaje-Automatico/blob/main/Practica_Miercoles_Tema8_RFM_Gabriel_Elizondo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Práctica del miércoles - Caso RFM

**Nombre:** Gabriel Elizondo Martinez  
**Matrícula:** AL07009102  
**Tema 8:** Clustering aplicado a segmentación RFM

## Reto 1. Filtrado y preparación de transacciones

Primero eliminé los registros sin `CustomerID`, las devoluciones representadas por cantidades negativas y los precios inválidos. Después calculé el monto de cada línea multiplicando cantidad por precio unitario.

In [1]:
import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

# create the raw transaction data
df_raw = pd.DataFrame({
    "CustomerID": [101, 102, 101, None, 103],
    "Quantity": [2, -1, 5, 3, 1],
    "UnitPrice": [10, 50, 20, 15, 100],
})

# remove missing customers, returns, and invalid prices
df_clean = df_raw.dropna(subset=["CustomerID"]).copy()
df_clean = df_clean[
    (df_clean["Quantity"] > 0) &
    (df_clean["UnitPrice"] > 0)
].copy()

# calculate the amount of each valid line
df_clean["CustomerID"] = df_clean["CustomerID"].astype(int)
df_clean["TotalPrice"] = df_clean["Quantity"] * df_clean["UnitPrice"]

display(df_clean)

,CustomerID,Quantity,UnitPrice,TotalPrice
0,101,2,10,20
2,101,5,20,100
4,103,1,100,100


El resultado conserva tres compras válidas. Se eliminaron el cliente 102 porque su cantidad era negativa y la fila sin identificador porque no podía asignarse a un cliente.

## Reto 2. Cálculo manual de Recency, Frequency y Monetary

Utilicé como fecha de referencia el 10 de enero de 2026. Recency corresponde a los días desde la última compra, Frequency al número de compras y Monetary al gasto acumulado.

In [2]:
# define purchase dates, amounts, and reference date
purchase_dates = pd.to_datetime([
    "2026-01-03",
    "2026-01-06",
    "2026-01-08",
])
purchase_amounts = pd.Series([100, 200, 90])
reference_date = pd.Timestamp("2026-01-10")

# calculate the three rfm metrics
recency = (reference_date - purchase_dates.max()).days
frequency = purchase_dates.nunique()
monetary = purchase_amounts.sum()

manual_rfm = pd.DataFrame({
    "Recency": [recency],
    "Frequency": [frequency],
    "Monetary": [monetary],
})

display(manual_rfm)

,Recency,Frequency,Monetary
0,2,3,390


Obtuve **Recency = 2 días**, **Frequency = 3 compras** y **Monetary = $390**. La última compra fue el 8 de enero, dos días antes de la fecha de referencia.

## Reto 3. Transformación logarítmica y escalado

Frequency y Monetary pueden tener valores muy dispersos. Apliqué `log1p` para reducir la influencia de valores muy altos y después utilicé `StandardScaler` para dejar las tres variables en escalas comparables.

In [3]:
# create the rfm example
rfm_data = pd.DataFrame({
    "Recency": [2, 45, 120, 5],
    "Frequency": [15, 2, 1, 8],
    "Monetary": [4500, 150, 50, 1200],
})

# apply log transformation to skewed variables
rfm_log = rfm_data.copy()
rfm_log["Frequency"] = np.log1p(rfm_log["Frequency"])
rfm_log["Monetary"] = np.log1p(rfm_log["Monetary"])

# standardize the transformed data
scaler = StandardScaler()
x_scaled_example = scaler.fit_transform(rfm_log)
x_scaled_example = pd.DataFrame(
    x_scaled_example,
    columns=rfm_log.columns,
)

print("Datos transformados:")
display(rfm_log.round(4))

print("Datos transformados y escalados:")
display(x_scaled_example.round(4))

Datos transformados:


,Recency,Frequency,Monetary
0,2,2.7726,8.4121
1,45,1.0986,5.0173
2,120,0.6931,3.9318
3,5,2.1972,7.0909


Datos transformados y escalados:


,Recency,Frequency,Monetary
0,-0.8616,1.2998,1.3164
1,0.0420,-0.7108,-0.6274
2,1.6181,-1.1978,-1.2489
3,-0.7985,0.6088,0.5599


Después del escalado, cada variable quedó centrada alrededor de cero. Esto evita que Monetary domine las distancias de K-Means solamente porque utiliza cantidades numéricas mayores.

## Reto 4. Comparación de Silhouette Score

El ejemplo anterior tiene únicamente cuatro clientes, por lo que no permite evaluar correctamente `k=4` y `k=5`. Para realizar la comparación solicitada utilicé 20 clientes simulados con distintos comportamientos RFM.

In [4]:
# create enough customers to compare k=3, k=4, and k=5
rfm_customers = pd.DataFrame({
    "CustomerID": range(1001, 1021),
    "Recency": [
        2, 3, 5, 7, 8,
        150, 180, 120, 200, 160,
        30, 45, 60, 35, 50,
        5, 10, 15, 8, 12,
    ],
    "Frequency": [
        20, 18, 25, 15, 22,
        1, 2, 1, 3, 2,
        8, 6, 5, 9, 7,
        2, 3, 1, 4, 2,
    ],
    "Monetary": [
        4000, 3500, 5000, 3000, 4500,
        80, 150, 50, 200, 120,
        900, 700, 600, 1100, 800,
        1800, 2200, 1500, 2500, 2000,
    ],
})

# transform and scale the modeling variables
rfm_model = rfm_customers[["Recency", "Frequency", "Monetary"]].copy()
rfm_model["Frequency"] = np.log1p(rfm_model["Frequency"])
rfm_model["Monetary"] = np.log1p(rfm_model["Monetary"])
x_scaled = StandardScaler().fit_transform(rfm_model)

# compare the requested values of k
silhouette_results = []
k_models = {}

for k in [3, 4, 5]:
    model = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = model.fit_predict(x_scaled)
    score = silhouette_score(x_scaled, labels)
    k_models[k] = model
    silhouette_results.append({
        "k": k,
        "Silhouette Score": score,
    })

silhouette_table = pd.DataFrame(silhouette_results)
best_k = int(
    silhouette_table.loc[
        silhouette_table["Silhouette Score"].idxmax(),
        "k",
    ]
)

display(silhouette_table.round(4))
print(f"Mejor valor: k={best_k}")

,k,Silhouette Score
0,3,0.6205
1,4,0.6888
2,5,0.6412


Mejor valor: k=4


El mejor resultado fue **k=4**, con un Silhouette Score de **0.6888**. Este valor fue mayor que los obtenidos con `k=3` y `k=5`, por lo que elegí cuatro clusters.

## Reto 5. Perfiles de negocio RFM

Asigné los clusters del modelo seleccionado al DataFrame original y calculé las medianas de Recency, Frequency y Monetary, además del número de clientes de cada grupo.

In [5]:
# assign the labels from the best model
best_model = k_models[best_k]
rfm_customers["Cluster"] = best_model.labels_

# calculate the requested cluster profiles
rfm_profiles = rfm_customers.groupby("Cluster").agg(
    Recency=("Recency", "median"),
    Frequency=("Frequency", "median"),
    Monetary=("Monetary", "median"),
    Count=("CustomerID", "count"),
)

# add a business name based on the observed medians
profile_names = {
    0: "Inactivos de bajo valor",
    1: "Clientes regulares",
    2: "VIP activos",
    3: "Ocasionales de alto ticket",
}
rfm_profiles["Perfil"] = rfm_profiles.index.map(profile_names)

display(rfm_profiles)

,Recency,Frequency,Monetary,Count,Perfil
Cluster,,,,,
0,160.0,2.0,120.0,5,Inactivos de bajo valor
1,45.0,7.0,800.0,5,Clientes regulares
2,5.0,20.0,4000.0,5,VIP activos
3,10.0,2.0,2000.0,5,Ocasionales de alto ticket


### Interpretación

- **Cluster 0 - Inactivos de bajo valor:** llevan mucho tiempo sin comprar, tienen poca frecuencia y gasto bajo. Conviene aplicar una campaña de reactivación.
- **Cluster 1 - Clientes regulares:** presentan valores intermedios en las tres métricas. Conviene trabajar retención y ventas adicionales.
- **Cluster 2 - VIP activos:** compraron recientemente, tienen frecuencia alta y el mayor gasto. Conviene ofrecer beneficios de fidelización.
- **Cluster 3 - Ocasionales de alto ticket:** compran pocas veces, pero gastan cantidades altas. Conviene dar seguimiento personalizado para aumentar su frecuencia.